<a href="https://colab.research.google.com/github/ChaconLima/mestrado/blob/programacao-binaria/Data_Envelopment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DEA - Data Envelopment Analysis**

Implementação disponível em: https://github.com/metjush/envelopment-py

In [1]:
"""
Data Envelopment Analysis implementation
Sources:
Sherman & Zhu (2006) Service Productivity Management, Improving Service Performance using Data Envelopment Analysis (DEA) [Chapter 2]
ISBN: 978-0-387-33211-6
http://deazone.com/en/resources/tutorial
"""

# Execute esta célula para compilar a função dea(X,Y)
# Modelo CCR orientado a inputs

import numpy as np
from scipy.optimize import fmin_slsqp


class DEA(object):

    def __init__(self, inputs, outputs):
        """
        Initialize the DEA object with input data
        n = number of entities (observations)
        m = number of inputs (variables, features)
        r = number of outputs
        :param inputs: inputs, n x m numpy array
        :param outputs: outputs, n x r numpy array
        :return: self
        """

        # supplied data
        self.inputs = inputs
        self.outputs = outputs

        # parameters
        self.n = inputs.shape[0]
        self.m = inputs.shape[1]
        self.r = outputs.shape[1]

        # iterators
        self.unit_ = range(self.n)
        self.input_ = range(self.m)
        self.output_ = range(self.r)

        # result arrays
        self.output_w = np.zeros((self.r, 1), dtype=np.float)  # output weights
        self.input_w = np.zeros((self.m, 1), dtype=np.float)  # input weights
        self.lambdas = np.zeros((self.n, 1), dtype=np.float)  # unit efficiencies
        self.efficiency = np.zeros_like(self.lambdas)  # thetas

        # names
        self.names = []

    def __efficiency(self, unit):
        """
        Efficiency function with already computed weights
        :param unit: which unit to compute for
        :return: efficiency
        """

        # compute efficiency
        denominator = np.dot(self.inputs, self.input_w)
        numerator = np.dot(self.outputs, self.output_w)

        return (numerator/denominator)[unit]

    def __target(self, x, unit):
        """
        Theta target function for one unit
        :param x: combined weights
        :param unit: which production unit to compute
        :return: theta
        """
        in_w, out_w, lambdas = x[:self.m], x[self.m:(self.m+self.r)], x[(self.m+self.r):]  # unroll the weights
        denominator = np.dot(self.inputs[unit], in_w)
        numerator = np.dot(self.outputs[unit], out_w)

        return numerator/denominator

    def __constraints(self, x, unit):
        """
        Constraints for optimization for one unit
        :param x: combined weights
        :param unit: which production unit to compute
        :return: array of constraints
        """

        in_w, out_w, lambdas = x[:self.m], x[self.m:(self.m+self.r)], x[(self.m+self.r):]  # unroll the weights
        constr = []  # init the constraint array

        # for each input, lambdas with inputs
        for input in self.input_:
            t = self.__target(x, unit)
            lhs = np.dot(self.inputs[:, input], lambdas)
            cons = t*self.inputs[unit, input] - lhs
            constr.append(cons)

        # for each output, lambdas with outputs
        for output in self.output_:
            lhs = np.dot(self.outputs[:, output], lambdas)
            cons = lhs - self.outputs[unit, output]
            constr.append(cons)

        # for each unit
        for u in self.unit_:
            constr.append(lambdas[u])

        return np.array(constr)

    def __optimize(self):
        """
        Optimization of the DEA model
        Use: http://docs.scipy.org/doc/scipy-0.17.0/reference/generated/scipy.optimize.linprog.html
        A = coefficients in the constraints
        b = rhs of constraints
        c = coefficients of the target function
        :return:
        """
        d0 = self.m + self.r + self.n
        # iterate over units
        for unit in self.unit_:
            # weights
            x0 = np.random.rand(d0) - 0.5
            x0 = fmin_slsqp(self.__target, x0, f_ieqcons=self.__constraints, args=(unit,))
            # unroll weights
            self.input_w, self.output_w, self.lambdas = x0[:self.m], x0[self.m:(self.m+self.r)], x0[(self.m+self.r):]
            self.efficiency[unit] = self.__efficiency(unit)

    def name_units(self, names):
        """
        Provide names for units for presentation purposes
        :param names: a list of names, equal in length to the number of units
        :return: nothing
        """

        assert(self.n == len(names))

        self.names = names

    def fit(self):
        """
        Optimize the dataset, generate basic table
        :return: table
        """

        self.__optimize()  # optimize

        print("Final thetas for each DMU:\n")
        print("---------------------------\n")
        for n, eff in enumerate(self.efficiency):
            if len(self.names) > 0:
                name = "DMU: %s" % self.names[n]
            else:
                name = "DMU: %d" % (n+1)
            print("%s theta: %.4f" % (name, eff))
            print("\n")
        print("---------------------------\n")

**Entrada de dados**

In [29]:
import pandas as pd
import numpy as np


dataframe = pd.read_csv('PL-DEA-ANATEL-Telecomunicacoes.csv')

# Entre com a matriz de Inputs
vetor1 = np.array(dataframe['Entradas_L'])
vetor2 = np.array(dataframe['Entradas _PT'])
vetor3 = np.array(dataframe['Entradas_AI'])
X = np.column_stack((vetor1,vetor2,vetor3))

# Entre com a matriz de Outputs
vetor1 = np.array(dataframe['Saídas _MN'])
vetor2 = np.array(dataframe['Saídas_P'])
vetor3 = np.array(dataframe['Saídas_AS'])
Y = np.column_stack((vetor1,vetor2,vetor3))

# Entre com os nomes das DMUs em forma de lista
names = np.array(dataframe['Empresa'])

print(X,Y,names)

[[   13707    99951  3692804]
 [   10947    73407  2895328]
 [    2373     7465   464154]
 [    1837     1669   561042]
 [    4785    54439  1406159]
 [     314     6776   170519]
 [     256    11681    25135]
 [    2821    41304   831171]
 [     686    13519   328803]
 [     556    12607   329721]
 [     303    34874   791541]
 [     649    10554    24633]
 [     868    15296    32177]
 [     105    23521   532904]
 [     220     2055     7147]
 [    1039     1042   315052]
 [     183     1602     4812]
 [    3461    25623  1193985]
 [   10659    46327  2227874]
 [     851     2203   154499]
 [    2633     1055   472702]
 [      44      163     7788]
 [     195    13745   451478]
 [    4859    38487  1155173]
 [      86      588    30402]
 [    3278    20175   884852]
 [     718     6345   253011]
 [     313     2924    93604]
 [    9731    53347  2101056]
 [     469     2015   120935]
 [    4955   223445 11185983]
 [    1307     3017   217837]
 [     374     2784   209829]
 [    5294

In [30]:
#Crie o problema dea através do comando:
dea = DEA(X,Y)
#Inclua o nome das DMUs através do comando:
dea.name_units(names)

<ipython-input-1-edb43866975b>:44: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  self.output_w = np.zeros((self.r, 1), dtype=np.float)  # output weights
<ipython-input-1-edb43866975b>:45: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  self.input_w = np.zeros((self.m, 1), dtype=np.float)  # input weights
<ipython-input-1-edb43866975b>:46: D

In [31]:
#Resolva os PLs e imprima os valores das eficiências:
dea.fit();

Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.8196780878893468
            Iterations: 46
            Function evaluations: 1886
            Gradient evaluations: 46
Positive directional derivative for linesearch    (Exit mode 8)
            Current function value: 0.8032377897863875
            Iterations: 56
            Function evaluations: 2193
            Gradient evaluations: 52
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.7691843071162094
            Iterations: 15
            Function evaluations: 615
            Gradient evaluations: 15
Optimization terminated successfully    (Exit mode 0)
            Current function value: 1.0000000216014069
            Iterations: 15
            Function evaluations: 617
            Gradient evaluations: 15
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.698630453957158
            Iterations: 26
         

In [12]:
# Entre com a matriz de Inputs
X = np.array([[5,14],
       [8,15],
       [7,12]])

# Entre com a matriz de Inputs
Y = np.array(
      [[9,4,16],
       [5,7,10],
       [4,9,13]])

# Entre com os nomes das DMUs em forma de lista
names = ['H1', 'H2', 'H3']

In [13]:
#Crie o problema dea através do comando:
dea = DEA(X,Y)
#Inclua o nome das DMUs através do comando:
dea.name_units(names)
#Resolva os PLs e imprima os valores das eficiências:
dea.fit();

Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.9999999967429666
            Iterations: 5
            Function evaluations: 46
            Gradient evaluations: 5
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.7733333567303463
            Iterations: 5
            Function evaluations: 45
            Gradient evaluations: 5
Optimization terminated successfully    (Exit mode 0)
            Current function value: 1.0000000259100128
            Iterations: 8
            Function evaluations: 72
            Gradient evaluations: 8
Final thetas for each DMU:

---------------------------

DMU: H1 theta: 1.0000


DMU: H2 theta: 0.7733


DMU: H3 theta: 1.0000


---------------------------



<ipython-input-1-edb43866975b>:44: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  self.output_w = np.zeros((self.r, 1), dtype=np.float)  # output weights
<ipython-input-1-edb43866975b>:45: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  self.input_w = np.zeros((self.m, 1), dtype=np.float)  # input weights
<ipython-input-1-edb43866975b>:46: D